In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%capture
!pip install torch-geometric

In [3]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU: NVIDIA L4
VRAM: 22.0 GB


In [4]:
import os
DATA_DIR = "/content/drive/MyDrive/5-core_AmazonElectronics"

files = ["train.parquet", "val.parquet", "test.parquet", "id_mappings.json"]


Load Data

In [5]:
import json
import pandas as pd

train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")
val_df   = pd.read_parquet(f"{DATA_DIR}/val.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}/test.parquet")

with open(f"{DATA_DIR}/id_mappings.json") as f:
    mappings = json.load(f)

n_users  = mappings["n_users"]
n_items  = mappings["n_items"]
user2idx = mappings["user2idx"]
item2idx = mappings["item2idx"]

print(f"train : {len(train_df):,}")
print(f"val   : {len(val_df):,}")
print(f"test  : {len(test_df):,}")
print(f"n_users = {n_users:,}  |  n_items = {n_items:,}")


train : 8,260,845
val   : 1,143,887
test  : 1,141,932
n_users = 1,145,516  |  n_items = 283,184


Model

In [6]:
import torch
import torch.nn as nn
from torch_geometric.nn import LGConv

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers

        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

        self.convs = nn.ModuleList([LGConv() for _ in range(n_layers)])

    def build_graph(self, train_df, device):
        users = torch.tensor(train_df["user_idx"].values, dtype=torch.long)
        items = torch.tensor(train_df["item_idx"].values, dtype=torch.long) + self.n_users

        src = torch.cat([users, items]).to(device)
        dst = torch.cat([items, users]).to(device)

        self.edge_index = torch.stack([src, dst], dim=0)

        # Tính degree
        num_nodes = self.n_users + self.n_items
        deg = torch.zeros(num_nodes, device=device)
        deg.scatter_add_(0, src, torch.ones(src.size(0), device=device))

        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0

        self.edge_weight = deg_inv_sqrt[src] * deg_inv_sqrt[dst]

    def forward(self):
        x = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        layers = [x]
        for conv in self.convs:
            x = conv(x, self.edge_index, self.edge_weight)
            layers.append(x)
        final = torch.stack(layers, dim=1).mean(dim=1)
        return final[:self.n_users], final[self.n_users:]

    def bpr_loss(self, users, pos_items, neg_items):
        # Forward để lấy final embeddings cho tính score
        user_emb, item_emb = self.forward()
        u   = user_emb[users]
        pos = item_emb[pos_items]
        neg = item_emb[neg_items]

        pos_score = (u * pos).sum(-1)
        neg_score = (u * neg).sum(-1)
        loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()


        u_0   = self.user_emb.weight[users]
        pos_0 = self.item_emb.weight[pos_items]
        neg_0 = self.item_emb.weight[neg_items]
        reg = (u_0.norm(2)**2 + pos_0.norm(2)**2 + neg_0.norm(2)**2) / len(users)

        return loss, reg

BPR Dataset

In [7]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

class BPRDataset(Dataset):
    def __init__(self, train_df, n_items):
        self.n_items  = n_items
        self.pairs    = train_df[["user_idx", "item_idx"]].values
        # User → set positive items (để tránh sample negative trùng)
        self.user_pos = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, pos = int(self.pairs[idx, 0]), int(self.pairs[idx, 1])
        while True:
            neg = np.random.randint(0, self.n_items)
            if neg not in self.user_pos[u]:
                break
        return torch.tensor(u), torch.tensor(pos), torch.tensor(neg)

dataset = BPRDataset(train_df, n_items)
u, p, n = dataset[1]
print(f"Sample: user={u.item()}, pos={p.item()}, neg={n.item()}")
print(f"Dataset size: {len(dataset):,}")

Sample: user=0, pos=12346, neg=75641
Dataset size: 8,260,845


Evaluation

In [8]:
@torch.no_grad()
def evaluate(model, eval_df, train_df, topk, device, batch_size=2048):
    """
    Full ranking evaluation — rank positive trên toàn bộ item catalog.
    Không dùng sampled negatives, tránh hoàn toàn bug negative sampling.
    """
    model.eval()
    user_emb, item_emb = model.forward()  # (n_users, D), (n_items, D)

    train_history = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

    users   = eval_df["user_idx"].values
    targets = eval_df["item_idx"].values

    recall_list, ndcg_list = [], []

    for start in range(0, len(users), batch_size):
        batch_users   = users[start: start + batch_size]
        batch_targets = targets[start: start + batch_size]

        u_t    = torch.tensor(batch_users, dtype=torch.long, device=device)
        u_emb  = user_emb[u_t]                   # (B, D)
        scores = u_emb @ item_emb.T              # (B, n_items)

        # Mask train positives (không rank item đã thấy trong train)
        for i, u in enumerate(batch_users):
            train_pos = list(train_history.get(int(u), set()))
            if train_pos:
                scores[i, train_pos] = -1e9

        # Rank của target trên toàn bộ n_items
        target_t      = torch.tensor(batch_targets, dtype=torch.long, device=device)
        target_scores = scores[torch.arange(len(batch_users), device=device), target_t]  # (B,)
        ranks         = (scores > target_scores.unsqueeze(1)).sum(dim=1) + 1             # (B,)

        hits = (ranks <= topk).float()
        recall_list.append(hits)

        ndcg = torch.where(
            ranks <= topk,
            1.0 / torch.log2(ranks.float() + 1),
            torch.zeros_like(ranks, dtype=torch.float)
        )
        ndcg_list.append(ndcg)

    recall = torch.cat(recall_list).mean().item()
    ndcg   = torch.cat(ndcg_list).mean().item()
    return recall, ndcg

Config

In [9]:
DEVICE      = torch.device("cuda")
EMB_DIM     = 64
N_LAYERS    = 3
LR          = 4e-3
WEIGHT_DECAY= 1e-4 # 1e-3
N_EPOCHS    = 20
BATCH_SIZE  = 8192
PATIENCE    = 10
TOPK        = 20
EVAL_EVERY  = 2

Init Model

In [10]:
import torch.optim as optim

model = LightGCN(n_users, n_items, EMB_DIM, N_LAYERS).to(DEVICE)
model.build_graph(train_df, DEVICE)

loader    = DataLoader(BPRDataset(train_df, n_items),
                       batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 91,436,800


Training loop

In [11]:
from tqdm import tqdm
import time

best_recall = 0.0
best_epoch  = 0
no_improve  = 0
OUTPUT_DIR  = "/content/lightgcn_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    pbar = tqdm(loader, desc=f"Epoch {epoch:3d}/{N_EPOCHS}",
                leave=False, dynamic_ncols=True)

    for users, pos_items, neg_items in pbar:
        users, pos_items, neg_items = (
            users.to(DEVICE), pos_items.to(DEVICE), neg_items.to(DEVICE)
        )
        optimizer.zero_grad()
        loss, reg = model.bpr_loss(users, pos_items, neg_items)
        (loss + WEIGHT_DECAY * reg).backward()
        optimizer.step()
        total_loss += loss.item()

        # Cập nhật loss realtime trên thanh progress
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(loader)

    if epoch % EVAL_EVERY == 0 or epoch == N_EPOCHS:
        recall, ndcg = evaluate(model, val_df, train_df, TOPK, DEVICE)
        improved = recall > best_recall

        if improved:
            best_recall = recall
            best_epoch  = epoch
            no_improve  = 0
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")
        else:
            no_improve += EVAL_EVERY

        marker = " ← best" if improved else ""
        print(f"Epoch {epoch:3d} | loss={avg_loss:.4f} | "
              f"Recall@{TOPK}={recall:.4f} | NDCG@{TOPK}={ndcg:.4f} | "
              f"{time.time()-t0:.1f}s{marker}")

        if no_improve >= PATIENCE:
            print(f"\nEarly stopping! Best Recall@{TOPK}={best_recall:.4f} @ epoch {best_epoch}")
            break
    else:
        print(f"Epoch {epoch:3d} | loss={avg_loss:.4f} | {time.time()-t0:.1f}s")

Epoch   1 | loss=0.3311 | 653.0s


Epoch   2 | loss=0.1494 | Recall@20=0.0334 | NDCG@20=0.0143 | 776.8s ← best


Epoch   3 | loss=0.0923 | 652.7s


Epoch   4 | loss=0.0660 | Recall@20=0.0347 | NDCG@20=0.0147 | 775.1s ← best


Epoch   5 | loss=0.0507 | 652.6s


...

Epoch  14 | loss=0.0158 | Recall@20=0.0349 | NDCG@20=0.0150 | 779.5s


Epoch  15 | loss=0.0147 | 652.6s


Epoch  16 | loss=0.0139 | Recall@20=0.0348 | NDCG@20=0.0151 | 782.5s


Epoch  17 | loss=0.0131 | 653.5s


Epoch  18 | loss=0.0126 | Recall@20=0.0343 | NDCG@20=0.0149 | 785.3s

Early stopping! Best Recall@20=0.0356 @ epoch 8


Test model

In [12]:
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pt"))

test_recall, test_ndcg = evaluate(model, test_df, train_df, TOPK, DEVICE)
print(f"\n{'='*45}")
print(f"  TEST  Recall@{TOPK} : {test_recall:.4f}")
print(f"  TEST  NDCG@{TOPK}   : {test_ndcg:.4f}")
print(f"{'='*45}")


  TEST  Recall@20 : 0.0243
  TEST  NDCG@20   : 0.0102


In [13]:
import shutil
shutil.copy(f"{OUTPUT_DIR}/best_model.pt",
            f"{DATA_DIR}/lightgcn_best.pt")
print("Saved to Drive")

Saved to Drive
